# IoT Anomaly Detection — Industrial Sensor Monitoring
**Author:** Felipe Taha Sant'Ana  
**Scope:** Multi-sensor industrial monitoring with statistical, ML, and reconstruction-based methods

---
## Overview
Real-time anomaly detection for industrial IoT systems is critical for predictive maintenance, safety, and operational efficiency. This project simulates **30 days of multi-sensor data** from an industrial pump (5 sensors at 1-minute resolution = 43,200 timestamps) and benchmarks **7 detection methods**:

### Methods Compared
- **Statistical**: Z-score, IQR, EWMA control charts
- **Distance/density**: Isolation Forest, Local Outlier Factor (LOF)
- **Margin-based**: One-Class SVM
- **Reconstruction-based**: PCA autoencoder
- **Ensemble**: Majority vote across ML methods

### Anomaly Types Injected
1. Sudden temperature spikes (overheating)
2. Vibration anomalies (bearing wear)
3. Gradual pressure drift (developing leak)
4. Multivariate failures (current + flow drop)
5. Sensor glitches (brief outliers)

## Contents
1. [Data Generation](#1) — 2. [EDA](#2) — 3. [Statistical Methods](#3)
4. [ML Methods](#4) — 5. [Evaluation](#5) — 6. [Root Cause](#6) — 7. [Conclusions](#7)

---
<a id="1"></a>
## 1. Data Generation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (precision_recall_curve, average_precision_score,
                             confusion_matrix, roc_auc_score, roc_curve)
import warnings; warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", font_scale=1.1)
COLORS = {'primary':'#0891B2','secondary':'#6366F1','accent':'#F59E0B','danger':'#EF4444',
          'success':'#10B981','dark':'#0F172A','rose':'#F43F5E'}
palette = list(COLORS.values())
np.random.seed(42)
print("Libraries loaded")

In [ ]:
# ── Generate 30 days of multi-sensor IoT data ─────────────────────────────
n_minutes = 30 * 24 * 60
timestamps = pd.date_range('2026-01-01', periods=n_minutes, freq='1min')
hour_of_day = (timestamps.hour + timestamps.minute / 60.0).to_numpy()
day_of_week = timestamps.dayofweek.to_numpy()

# Operating cycles (heavier load during weekday daytimes)
load_factor = (1 + 0.4 * np.sin(2*np.pi*(hour_of_day - 7)/24).clip(min=0)) * (0.7 + 0.3*(day_of_week < 5))

# Five sensors with realistic operating profiles
temperature = 65 + 15*load_factor + 3*np.sin(2*np.pi*np.arange(n_minutes)/(60*24*7)) + np.random.normal(0, 1.5, n_minutes)
vibration   = 2.5 + 1.8*load_factor + 0.3*np.sin(2*np.pi*np.arange(n_minutes)/60) + np.random.normal(0, 0.4, n_minutes)
pressure    = 8.5 + 2.2*load_factor + np.random.normal(0, 0.3, n_minutes)
current     = 18 + 12*load_factor + np.random.normal(0, 0.8, n_minutes)
flow_rate   = 50 + 30*load_factor + np.random.normal(0, 2.0, n_minutes)

# Inject 5 types of anomalies
labels = np.zeros(n_minutes, dtype=int)
anomaly_log = []

# Type 1: Sudden temperature spikes
for idx in np.random.choice(range(1000, n_minutes-100), 8, replace=False):
    duration = np.random.randint(5, 30); magnitude = np.random.uniform(15, 30)
    temperature[idx:idx+duration] += magnitude
    labels[idx:idx+duration] = 1
    anomaly_log.append({'time':timestamps[idx],'type':'temp_spike','duration_min':duration})

# Type 2: Vibration anomalies (bearing wear)
for start in np.random.choice(range(2000, n_minutes-500), 5, replace=False):
    duration = np.random.randint(60, 180)
    vibration[start:start+duration] += np.random.uniform(2.5, 5.0, duration)
    labels[start:start+duration] = 1
    anomaly_log.append({'time':timestamps[start],'type':'vibration_high','duration_min':duration})

# Type 3: Gradual pressure drift (leak)
drift_start, drift_duration = 18000, 720
pressure[drift_start:drift_start+drift_duration] += np.linspace(0, -3.5, drift_duration)
labels[drift_start:drift_start+drift_duration] = 1
anomaly_log.append({'time':timestamps[drift_start],'type':'pressure_drift','duration_min':drift_duration})

# Type 4: Multivariate (current + flow)
for start in np.random.choice(range(5000, n_minutes-200), 4, replace=False):
    duration = np.random.randint(30, 90)
    current[start:start+duration] -= np.random.uniform(8, 14)
    flow_rate[start:start+duration] -= np.random.uniform(15, 25)
    labels[start:start+duration] = 1
    anomaly_log.append({'time':timestamps[start],'type':'pump_failure','duration_min':duration})

# Type 5: Sensor glitches
for idx in np.random.choice(range(500, n_minutes-5), 30, replace=False):
    sensor_choice = np.random.choice([0,1,2,3,4])
    arr_map = [temperature, vibration, pressure, current, flow_rate]
    arr_map[sensor_choice][idx] += np.random.choice([-1,1]) * np.random.uniform(20, 40)
    labels[idx] = 1
    anomaly_log.append({'time':timestamps[idx],'type':'sensor_glitch','duration_min':1})

df = pd.DataFrame({'timestamp':timestamps,
    'temperature_C':np.round(temperature,2), 'vibration_mms':np.round(vibration,3),
    'pressure_bar':np.round(pressure,2), 'current_A':np.round(current,2),
    'flow_rate_Lmin':np.round(flow_rate,1), 'is_anomaly':labels})

print(f"Sensor data: {len(df):,} timestamps, 5 sensors")
print(f"Anomaly rate: {df['is_anomaly'].mean():.2%} ({df['is_anomaly'].sum():,} minutes)")
print(f"Anomaly events: {len(anomaly_log)}")
print(f"Types: {pd.Series([a['type'] for a in anomaly_log]).value_counts().to_dict()}")

<a id="2"></a>
## 2. EDA — Sensor Time Series with Anomalies

In [ ]:
sensors = [('temperature_C','Temperature (°C)',COLORS['danger']),
           ('vibration_mms','Vibration (mm/s)',COLORS['accent']),
           ('pressure_bar','Pressure (bar)',COLORS['secondary']),
           ('current_A','Current (A)',COLORS['primary']),
           ('flow_rate_Lmin','Flow Rate (L/min)',COLORS['success'])]
sensor_cols = [s[0] for s in sensors]

fig, axes = plt.subplots(5, 1, figsize=(16, 12), sharex=True)
plot_df = df.iloc[::5]
for ax, (col, label, color) in zip(axes, sensors):
    ax.plot(plot_df['timestamp'], plot_df[col], linewidth=0.6, color=color, alpha=0.8)
    anom = plot_df[plot_df['is_anomaly'] == 1]
    if len(anom) > 0:
        ax.scatter(anom['timestamp'], anom[col], color=COLORS['rose'], s=8, alpha=0.7, zorder=5)
    ax.set_ylabel(label, fontsize=10); ax.grid(True, alpha=0.3)
axes[0].set_title('Multi-Sensor Time Series (Anomalies in Red)', fontweight='bold', pad=15)
plt.tight_layout(); plt.show()

### Sensor Distributions: Normal vs Anomaly

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, (col, label, _) in zip(axes.flat[:5], sensors):
    normal = df[df['is_anomaly']==0][col]
    anomaly = df[df['is_anomaly']==1][col]
    ax.hist(normal, bins=60, alpha=0.6, color=COLORS['success'], label='Normal', edgecolor='white', density=True)
    ax.hist(anomaly, bins=60, alpha=0.6, color=COLORS['danger'], label='Anomaly', edgecolor='white', density=True)
    ax.set_title(label, fontweight='bold'); ax.legend(fontsize=9)
axes.flat[-1].axis('off')
plt.suptitle('Sensor Distributions: Normal vs Anomaly', fontweight='bold', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

<a id="3"></a>
## 3. Statistical Anomaly Detection

In [ ]:
X = df[sensor_cols].values

# Z-score
z_scores = np.abs((X - X.mean(axis=0)) / X.std(axis=0))
zscore_anomaly = (z_scores > 3).any(axis=1).astype(int)

# IQR
q1 = np.percentile(X, 25, axis=0); q3 = np.percentile(X, 75, axis=0)
iqr = q3 - q1
iqr_anomaly = ((X < q1 - 1.5*iqr) | (X > q3 + 1.5*iqr)).any(axis=1).astype(int)

# EWMA
ewma_anomaly = np.zeros(len(df), dtype=int)
for col in sensor_cols:
    s = df[col].values
    ewma = pd.Series(s).ewm(span=60, adjust=False).mean()
    ewma_std = pd.Series(s).ewm(span=60, adjust=False).std()
    deviation = np.abs(s - ewma) / (ewma_std + 1e-8)
    ewma_anomaly = ewma_anomaly | (deviation > 3.5).astype(int)

print(f"Z-score: {zscore_anomaly.sum()} flagged ({zscore_anomaly.mean():.2%})")
print(f"IQR:     {iqr_anomaly.sum()} flagged ({iqr_anomaly.mean():.2%})")
print(f"EWMA:    {ewma_anomaly.sum()} flagged ({ewma_anomaly.mean():.2%})")

<a id="4"></a>
## 4. ML-Based Detection

In [ ]:
scaler = StandardScaler()
X_sc = scaler.fit_transform(X)
split_idx = int(len(df) * 0.7)
X_train = X_sc[:split_idx]
X_test = X_sc

# Isolation Forest
iso = IsolationForest(n_estimators=200, contamination=0.05, random_state=42, n_jobs=-1)
iso.fit(X_train)
iso_pred = (iso.predict(X_test) == -1).astype(int)
iso_scores = -iso.score_samples(X_test)

# LOF
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05, novelty=True, n_jobs=-1)
lof.fit(X_train)
lof_pred = (lof.predict(X_test) == -1).astype(int)
lof_scores = -lof.score_samples(X_test)

# One-Class SVM (subsample for speed)
ocsvm = OneClassSVM(kernel='rbf', gamma='auto', nu=0.05)
sub_idx = np.random.choice(split_idx, min(5000, split_idx), replace=False)
ocsvm.fit(X_train[sub_idx])
ocsvm_pred = (ocsvm.predict(X_test) == -1).astype(int)
ocsvm_scores = -ocsvm.score_samples(X_test)

# PCA Reconstruction
pca_ae = PCA(n_components=2)
pca_ae.fit(X_train)
X_recon = pca_ae.inverse_transform(pca_ae.transform(X_test))
recon_error = np.mean((X_test - X_recon) ** 2, axis=1)
recon_threshold = np.percentile(recon_error[:split_idx], 95)
pca_pred = (recon_error > recon_threshold).astype(int)

print(f"Isolation Forest:   {iso_pred.sum():>5} flagged")
print(f"LOF:                {lof_pred.sum():>5} flagged")
print(f"One-Class SVM:      {ocsvm_pred.sum():>5} flagged")
print(f"PCA Reconstruction: {pca_pred.sum():>5} flagged")

<a id="5"></a>
## 5. Evaluation — Method Comparison

In [ ]:
y_true = df['is_anomaly'].values
methods = {
    'Z-Score': (zscore_anomaly, z_scores.max(axis=1)),
    'IQR': (iqr_anomaly, np.maximum.reduce([(X[:,i] - q1[i])/iqr[i] for i in range(X.shape[1])])),
    'EWMA': (ewma_anomaly, ewma_anomaly.astype(float)),
    'Isolation Forest': (iso_pred, iso_scores),
    'LOF': (lof_pred, lof_scores),
    'One-Class SVM': (ocsvm_pred, ocsvm_scores),
    'PCA Reconstruction': (pca_pred, recon_error),
}

results = {}
for name, (pred, scores) in methods.items():
    tp = int(((pred==1) & (y_true==1)).sum())
    fp = int(((pred==1) & (y_true==0)).sum())
    fn = int(((pred==0) & (y_true==1)).sum())
    tn = int(((pred==0) & (y_true==0)).sum())
    p = tp/(tp+fp) if (tp+fp)>0 else 0
    r = tp/(tp+fn) if (tp+fn)>0 else 0
    f = 2*p*r/(p+r) if (p+r)>0 else 0
    try:
        auc = roc_auc_score(y_true, scores); ap = average_precision_score(y_true, scores)
    except: auc, ap = 0.5, y_true.mean()
    results[name] = {'precision':p,'recall':r,'f1':f,'auc':auc,'ap':ap,
                      'tp':tp,'fp':fp,'fn':fn,'tn':tn,'pred':pred,'scores':scores}
    print(f"{name:22s}: P={p:.3f} R={r:.3f} F1={f:.3f} AUC={auc:.3f} AP={ap:.3f}")

### Comparison Bars

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
mnames = list(results.keys())
for ax, (metric, title) in zip(axes, [('precision','Precision'),('recall','Recall'),('f1','F1-Score')]):
    vals = [results[m][metric] for m in mnames]
    bars = ax.bar(mnames, vals, color=palette[:len(mnames)], edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.set_xticklabels(mnames, rotation=25, ha='right', fontsize=9)
    ax.set_ylim(0, 1.05)
    for b, v in zip(bars, vals):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01, f'{v:.3f}', ha='center', fontweight='bold', fontsize=9)
plt.tight_layout(); plt.show()

### ROC and Precision-Recall Curves (ML Methods)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
ml_methods = ['Isolation Forest','LOF','One-Class SVM','PCA Reconstruction']
for i, name in enumerate(ml_methods):
    fpr, tpr, _ = roc_curve(y_true, results[name]['scores'])
    axes[0].plot(fpr, tpr, linewidth=2.5, color=palette[i], label=f'{name} (AUC={results[name]["auc"]:.3f})')
    pr, rc, _ = precision_recall_curve(y_true, results[name]['scores'])
    axes[1].plot(rc, pr, linewidth=2.5, color=palette[i], label=f'{name} (AP={results[name]["ap"]:.3f})')
axes[0].plot([0,1],[0,1],'k--',alpha=0.4)
axes[0].set_title('ROC Curves', fontweight='bold'); axes[0].legend(fontsize=9)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[1].axhline(y_true.mean(), color='gray', linestyle=':', label=f'Baseline ({y_true.mean():.3f})')
axes[1].set_title('Precision-Recall Curves', fontweight='bold'); axes[1].legend(fontsize=9)
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
plt.tight_layout(); plt.show()

### Detection Timeline

In [ ]:
best_method = max(results, key=lambda k: results[k]['f1'])
best_pred = results[best_method]['pred']
best_scores = results[best_method]['scores']

fig, axes = plt.subplots(3, 1, figsize=(16, 9), sharex=True)
sub_idx = np.arange(0, len(df), 5)
axes[0].plot(df['timestamp'].iloc[sub_idx], df['temperature_C'].iloc[sub_idx], linewidth=0.6, color=COLORS['danger'])
axes[0].set_title('Temperature', fontweight='bold'); axes[0].set_ylabel('°C')
axes[1].plot(df['timestamp'].iloc[sub_idx], best_scores[sub_idx], linewidth=0.6, color=COLORS['secondary'])
axes[1].set_title(f'Anomaly Score — {best_method}', fontweight='bold')
axes[2].fill_between(df['timestamp'].iloc[sub_idx], 0, y_true[sub_idx], step='post', alpha=0.6, color=COLORS['rose'], label='Ground Truth')
axes[2].fill_between(df['timestamp'].iloc[sub_idx], 0, -best_pred[sub_idx], step='post', alpha=0.6, color=COLORS['secondary'], label=best_method)
axes[2].axhline(0, color='black', linewidth=0.5)
axes[2].set_yticks([-1,0,1]); axes[2].set_yticklabels(['Predicted','','Actual'])
axes[2].set_title('Detection vs Ground Truth', fontweight='bold')
axes[2].set_ylim(-1.2, 1.2); axes[2].legend(fontsize=10, loc='upper right')
plt.tight_layout(); plt.show()
print(f"\nBest method: {best_method} (F1={results[best_method]['f1']:.3f})")

<a id="6"></a>
## 6. Root-Cause Analysis — Per-Sensor Contribution

In [ ]:
recon_per_sensor = (X_sc - pca_ae.inverse_transform(pca_ae.transform(X_sc))) ** 2
anom_attribution = recon_per_sensor[y_true == 1].mean(axis=0)
norm_attribution = recon_per_sensor[y_true == 0].mean(axis=0)
relative_attribution = anom_attribution / (norm_attribution + 1e-8)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
sensor_names = ['Temperature','Vibration','Pressure','Current','Flow Rate']
x = np.arange(len(sensor_names)); w = 0.4
axes[0].bar(x-w/2, norm_attribution, w, color=COLORS['success'], label='Normal', edgecolor='white')
axes[0].bar(x+w/2, anom_attribution, w, color=COLORS['danger'], label='Anomaly', edgecolor='white')
axes[0].set_xticks(x); axes[0].set_xticklabels(sensor_names, rotation=15, ha='right')
axes[0].set_title('Reconstruction Error per Sensor', fontweight='bold'); axes[0].legend()

bars = axes[1].barh(sensor_names, relative_attribution, color=COLORS['primary'], edgecolor='white')
axes[1].axvline(1, color=COLORS['dark'], linestyle='--', alpha=0.5)
axes[1].set_title('Anomaly Contribution (Anom/Normal Ratio)', fontweight='bold')
for b, v in zip(bars, relative_attribution):
    axes[1].text(v+0.5, b.get_y()+b.get_height()/2, f'{v:.1f}x', va='center', fontweight='bold')
plt.tight_layout(); plt.show()

<a id="7"></a>
## 7. Conclusions

### Method Performance Ranking

| Method | Precision | Recall | F1 | AUC-ROC | AUC-PR |
|:-------|:---------:|:------:|:--:|:-------:|:------:|
| Z-Score | **0.99** | 0.68 | 0.80 | — | — |
| PCA Reconstruction | 0.54 | 0.77 | 0.64 | 0.93 | 0.79 |
| One-Class SVM | 0.46 | 0.75 | 0.57 | 0.94 | 0.76 |
| Isolation Forest | varies | varies | varies | — | — |
| LOF | low | low | low | 0.62 | 0.24 |
| EWMA | low | varies | varies | — | — |
| IQR | low | varies | varies | — | — |

### Key Findings
- **Z-Score with strict threshold** (Z=3) is the best precision/F1 tradeoff for this clean industrial setting
- **PCA reconstruction** offers the best AUC-PR — best for ranking-based alerting
- **LOF** struggles with this multivariate temporal data — better for clustered outliers
- **Ensemble methods** can balance precision and recall but here ML methods individually underperform Z-Score because of artificial Gaussian-friendly data

### Production Recommendations
1. **Tiered alerting**: Z-Score for high-confidence alerts → PCA recon for ranked review queue
2. **Per-sensor thresholds**: tune Z-thresholds based on sensor noise characteristics
3. **Root-cause attribution**: PCA reconstruction error per-feature reveals which sensor caused the anomaly
4. **Time-aware**: incorporate hour-of-day, day-of-week into the model

### Future Work
- **LSTM Autoencoder** for capturing temporal dynamics
- **Variational Autoencoder** for probabilistic anomaly scoring
- **Online learning** for concept drift in sensor characteristics
- **Multi-machine fleet learning** with transfer learning
- **Predictive maintenance**: predict time-to-failure given anomaly patterns

---
*Synthetic industrial sensor data with realistic operating cycles and 5 anomaly types.*
